# Construct paired spectra and sampling plans

The visible preprocessing code builds compact replicate banks and index-based sampling plans. Inputs pair a clean spectrum with the requested Na/Ca/K/Mg condition; targets are measured interfered spectra with the same Cu/Ni/Zn concentration combination.

Training augmentation replaces metal-specific spectral windows using replicates from the same interference group. These constructed combinations are not independent experimental acquisitions. Test pairs use a selected clean replicate and an individual measured interfered replicate.

This notebook requires the missing shared module `spice_moe.py`, which supplies metadata, wavelength windows, and normalization settings.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ["MOE_PROJECT_ROOT"]) if "MOE_PROJECT_ROOT" in os.environ else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
from project_paths import configure_run
DATA_ROOT, RUN_ROOT, PREPARED_DIR, CHECKPOINT_DIR = configure_run(new=False)


In [ ]:
# === 01 / Step 0: paths and configuration ================================
# Builds the training "sampling plan": for every Na/Ca/K/Mg group we store only
# WHICH replicate rows to draw, plus a small bank of raw spectra.  The full space
# is 60 backgrounds x 60 Cu x 60 Ni x 60 Zn = 12,960,000 per group; storing the
# assembled spectra would be tens of GB, storing indices is a few hundred MB.
import os, sys, itertools
import numpy as np, pandas as pd, h5py
sys.path.insert(0, os.getcwd())
import spice_moe as S

DATA_DIR = str(DATA_ROOT)
OUT_DIR  = str(PREPARED_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

META_COLS, BKG_COLS, METAL_COLS = S.META_COLS, S.BKG_COLS, S.METAL_COLS
SAMPLES_PER_COMBO = 100
# Multi-ion conditions are re-sampled more often: interference saturates strongly
# (four ions at 500 ppm give only ~35% of the additive sum) and that rule can only
# be learned where multi-ion samples exist.
ION_GROUP_REPEAT = {0: 1, 1: 1, 2: 2, 3: 2, 4: 3}
_rng = np.random.default_rng(S.SEED)

RAW_DIRS = {
    "na_train": "20260128_MoE Na",   "ca_train": "20260128_MoE Ca",
    "k_train":  "20260128_MoE K",    "mg_train": "20260128_MoE Mg",
    "naca_train": "20260129_MoE Na+Ca",
    "nacak_train": "20260130_MoE Na, Ca + K",
    "nacakmg_train": "20260130_MoE Na, Ca, K + Mg",
    "testing": "20260130_MoE Testing dataset",
}
print("DATA_DIR =", DATA_DIR)


In [ ]:
# === 01 / Step 1: readers ================================================
def load_rows(path):
    df = pd.read_csv(path)
    num = [c for c in df.columns if c.replace('.', '', 1).isdigit()]
    num = [num[i] for i in np.argsort([float(c) for c in num])]
    wl = np.array([float(c) for c in num])
    return df, df[num].values.astype(np.float32), wl, num

def remove_pb(spec, wl):
    """Mask the Pb lines by linear interpolation across two intervals."""
    spec = spec.copy()
    for x0, x1 in [(405.24, 407.245), (367.942, 369.659)]:
        li = int(np.argmin(np.abs(wl - x0))); ui = int(np.argmin(np.abs(wl - x1)))
        if ui <= li: continue
        y0, y1 = spec[:, li].copy(), spec[:, ui].copy()
        for j in range(li, ui + 1):
            t = (wl[j] - wl[li]) / (wl[ui] - wl[li] + 1e-12)
            spec[:, j] = y0 * (1 - t) + y1 * t
    return spec

def load_concat(folder):
    df, spec, wl, num = load_rows(os.path.join(DATA_DIR, folder, "Concat.csv"))
    return df, remove_pb(spec, wl), wl, num

def load_combined(folder):
    """Concat.csv plus any older conditions that only exist in Mean.csv
    (many of those have K=0 or Mg=0 and must keep contributing to training)."""
    dfC, specC, wl, num = load_concat(folder)
    dfM, specM, wlM, _  = load_rows(os.path.join(DATA_DIR, folder, "Mean.csv"))
    specM = remove_pb(specM, wlM)
    assert len(wl) == len(wlM) and np.allclose(wl, wlM), f"wavelength mismatch in {folder}"
    keyC = set(map(tuple, np.round(dfC[BKG_COLS + METAL_COLS].values, 3)))
    ext = np.array([tuple(k) not in keyC
                    for k in np.round(dfM[BKG_COLS + METAL_COLS].values, 3)])
    meta = pd.concat([dfC[META_COLS], dfM[META_COLS][ext]], ignore_index=True)
    return meta, np.vstack([specC, specM[ext]]), wl, num, int(ext.sum())

def diag_level_pools(meta, spec, rows):
    """Within one interference group, collect replicates by the diagonal metal level."""
    pools = {}
    for i in rows:
        cu, ni, zn = meta['Cu'].iloc[i], meta['Ni'].iloc[i], meta['Zn'].iloc[i]
        if cu == ni == zn:
            pools.setdefault(int(cu), []).append(spec[i])
    return {k: np.stack(v) for k, v in pools.items()}


In [ ]:
# === 01 / Step 2: testing pairs + global clean pool =======================
def process_testing(folder):
    """Testing uses exactly the same construction as training and validation:
    a SINGLE clean replicate as X, a SINGLE measured replicate as the label.
    No averaging, no synthetic noise -- so the three splits are directly comparable."""
    dfC, specC, wl, num = load_rows(os.path.join(DATA_DIR, folder, "Concat.csv"))
    specC = remove_pb(specC, wl)
    pm = (dfC[BKG_COLS] == 0).all(axis=1).values
    pure = {}
    for i in np.where(pm)[0]:
        pure.setdefault(tuple(np.round(dfC[METAL_COLS].iloc[i].values, 3)), []).append(specC[i])
    pure = {k: np.stack(v) for k, v in pure.items()}
    rng = np.random.default_rng(S.SEED)
    X, Y, skip = [], [], 0
    for i in range(len(dfC)):
        combo = tuple(np.round(dfC[METAL_COLS].iloc[i].values, 3))
        if combo not in pure:
            skip += 1; continue
        x = pure[combo][rng.integers(len(pure[combo]))]
        m = dfC[META_COLS].iloc[i].values.astype(np.float32)
        X.append(np.concatenate([m, x])); Y.append(np.concatenate([m, specC[i]]))
    return (np.array(X, np.float32), np.array(Y, np.float32), num,
            f"testing single-replicate pairs={len(X)} skip={skip}")

# Global clean (Realp) pool, shared by every group so that a group missing a metal
# level can still be paired with a clean spectrum at that level.
GLOBAL_PURE = {}
for _k, _f in RAW_DIRS.items():
    if _k == "testing": continue
    _df, _sp, _wl, _num = load_concat(_f)
    _m = ((_df[BKG_COLS] == 0).all(axis=1).values
          & (_df['Cu'] == _df['Ni']).values & (_df['Ni'] == _df['Zn']).values)
    for _i in np.where(_m)[0]:
        GLOBAL_PURE.setdefault(int(_df['Cu'].iloc[_i]), []).append(_sp[_i])
GLOBAL_PURE = {k: np.stack(v) for k, v in GLOBAL_PURE.items()}
print("clean replicates per level:", {k: len(v) for k, v in sorted(GLOBAL_PURE.items())})


In [ ]:
# === 01 / Step 3: build the sampling plan per group ======================
def build_plan(folder):
    """For one raw folder return the plan:
        g_meta (G,12)   the condition of each Na/Ca/K/Mg group
        g_lv   (G,6,2)  row range of each metal level inside y_bank
        g_bg   (G,2)    row range of the whole group (background suppliers)
        g_w    (G,)     multi-ion resampling weight
        x_lv   (6,2)    row range of each metal level inside x_bank
        y_bank / x_bank raw spectra (small)
    A spectrum is assembled as: one background replicate, then the Zn / Ni / Cu
    segments overwritten from replicates at the requested level.  Cu, Ni and Zn
    bands are physically independent, so any (Cu, Ni, Zn) combination is reachable.
    Sampling NEVER mixes across Na/Ca/K/Mg groups."""
    meta, spec, wl, num, n_extra = load_combined(folder)
    zn_i = S.band_index(*S.ZN_RANGE, wl)
    ni_i = S.band_index(*S.NI_RANGE, wl)
    cu_i = S.band_index(*S.CU_RANGE, wl)

    pure_rows = np.where(((meta[BKG_COLS] == 0).all(axis=1)).values)[0]
    infile = diag_level_pools(meta, spec, pure_rows) if len(pure_rows) else {}
    pure_pools, used_global = {}, False
    for lv in sorted(set(list(infile.keys()) + list(GLOBAL_PURE.keys()))):
        own = infile.get(lv)
        if own is not None and len(own) >= 5:
            pure_pools[lv] = own
        else:
            parts = ([own] if own is not None else []) + \
                    ([GLOBAL_PURE[lv]] if lv in GLOBAL_PURE else [])
            if parts:
                pure_pools[lv] = np.concatenate(parts, 0); used_global = True
    x_levels = sorted(pure_pools)
    x_bank = np.concatenate([pure_pools[lv] for lv in x_levels], 0).astype(np.float32)
    x_rows, off = {}, 0
    for lv in x_levels:
        n = len(pure_pools[lv]); x_rows[lv] = np.arange(off, off + n); off += n

    y_chunks, y_rows_by_group, meta_by_group, off = [], [], [], 0
    for bkg_vals, idxs in meta.groupby(BKG_COLS).groups.items():
        rows = [meta.index.get_loc(i) for i in idxs]
        pools = diag_level_pools(meta, spec, rows)
        levels = sorted(set(pools) & set(pure_pools))
        if not levels: continue
        rows_lv = {}
        for lv in levels:
            n = len(pools[lv]); y_chunks.append(pools[lv].astype(np.float32))
            rows_lv[lv] = np.arange(off, off + n); off += n
        n_active = int(sum(1 for v in np.atleast_1d(bkg_vals) if float(v) > 0))
        y_rows_by_group.append((rows_lv, levels, n_active))
        meta_by_group.append(meta[META_COLS].iloc[rows[0]].values.astype(np.float32).copy())
    y_bank = np.concatenate(y_chunks, 0)

    # explicit index lists (kept for the validation split and for plotting)
    M, YI, XI = [], [], []
    x_all = np.arange(len(x_bank))
    for (rows_lv, levels, n_active), mt in zip(y_rows_by_group, meta_by_group):
        g_all = np.concatenate([rows_lv[lv] for lv in levels])
        n_draw = SAMPLES_PER_COMBO * ION_GROUP_REPEAT.get(n_active, 1)
        for cu, ni, zn in itertools.product(levels, repeat=3):
            m = mt.copy(); m[5], m[6], m[7] = cu, ni, zn
            M.append(np.repeat(m[None, :], n_draw, 0))
            YI.append(np.stack([_rng.choice(g_all, n_draw), _rng.choice(rows_lv[zn], n_draw),
                                _rng.choice(rows_lv[ni], n_draw), _rng.choice(rows_lv[cu], n_draw)], 1))
            XI.append(np.stack([_rng.choice(x_all, n_draw), _rng.choice(x_rows[zn], n_draw),
                                _rng.choice(x_rows[ni], n_draw), _rng.choice(x_rows[cu], n_draw)], 1))

    n_lv = 6
    g_lv = np.full((len(meta_by_group), n_lv, 2), -1, np.int32)
    g_bg = np.zeros((len(meta_by_group), 2), np.int32)
    g_w  = np.zeros(len(meta_by_group), np.int32)
    for gi, (rows_lv, levels, n_active) in enumerate(y_rows_by_group):
        allr = np.concatenate([rows_lv[lv] for lv in levels])
        g_bg[gi] = [allr.min(), allr.max() + 1]
        g_w[gi]  = ION_GROUP_REPEAT.get(n_active, 1)
        for lv in levels:
            g_lv[gi, lv] = [rows_lv[lv].min(), rows_lv[lv].max() + 1]
    x_lv = np.full((n_lv, 2), -1, np.int32)
    for lv in x_levels:
        x_lv[lv] = [x_rows[lv].min(), x_rows[lv].max() + 1]

    plan = dict(g_meta=np.stack(meta_by_group).astype(np.float32), g_lv=g_lv, g_bg=g_bg,
                g_w=g_w, x_lv=x_lv, meta=np.concatenate(M).astype(np.float32),
                y_idx=np.concatenate(YI).astype(np.int32),
                x_idx=np.concatenate(XI).astype(np.int32),
                y_bank=y_bank, x_bank=x_bank,
                bands=dict(Zn=zn_i, Ni=ni_i, Cu=cu_i), wl=wl, num=num)
    info = (f"samples={len(plan['meta']):,d}  groups={len(meta_by_group)}  "
            f"y_bank={y_bank.shape}  x_bank={x_bank.shape}  "
            f"extra_from_mean={n_extra}  used_global_clean={used_global}")
    return plan, info


In [ ]:
# === 01 / Step 4: run and save ===========================================
Nm = len(META_COLS); total = 0
for key, folder in RAW_DIRS.items():
    if key == "testing":
        X, Y, num, info = process_testing(folder)
        X[:, Nm:] = np.clip(X[:, Nm:], 0, S.SPEC_SCALE)
        Y[:, Nm:] = np.clip(Y[:, Nm:], 0, S.SPEC_SCALE)
        cols = META_COLS + num
        with h5py.File(os.path.join(OUT_DIR, f"SPICE_{key}_data.h5"), "w") as h:
            h.create_dataset("X_with_meta", data=X.astype(np.float32))
            h.create_dataset("X_columns", data=np.array(cols, dtype='S'))
        with h5py.File(os.path.join(OUT_DIR, f"SPICE_{key}_label.h5"), "w") as h:
            h.create_dataset("Y_with_meta", data=Y.astype(np.float32))
            h.create_dataset("Y_columns", data=np.array(cols, dtype='S'))
        pd.DataFrame(X, columns=cols).to_csv(os.path.join(OUT_DIR, f"SPICE_{key}_data.csv"), index=False)
        pd.DataFrame(Y, columns=cols).to_csv(os.path.join(OUT_DIR, f"SPICE_{key}_label.csv"), index=False)
        print(f"[{key:14s}] {info}")
        continue

    plan, info = build_plan(folder)
    p = os.path.join(OUT_DIR, f"SPICE_{key}_plan.h5")
    with h5py.File(p, "w") as h:
        for k in ("meta", "y_idx", "x_idx", "y_bank", "x_bank"):
            h.create_dataset(k, data=plan[k], compression="gzip", compression_opts=1)
        for k in ("g_meta", "g_lv", "g_bg", "g_w", "x_lv"):
            h.create_dataset(k, data=plan[k])
        h.create_dataset("wl", data=plan["wl"].astype(np.float32))
        h.create_dataset("columns", data=np.array(META_COLS + plan["num"], dtype='S'))
        for b, (lo, hi) in plan["bands"].items():
            h.attrs[b] = np.array([lo, hi], np.int32)
    total += len(plan["meta"])
    print(f"[{key:14s}] {info}")
    print(f"     -> {os.path.basename(p)}  ({os.path.getsize(p)/1e6:,.0f} MB)")

print(f"\nDone.  Indexed training samples = {total:,d}")
print("During training 02-04 draw fresh batches from the full 60^4 space, so these")
print("indices are only the validation split and the plotting subset.")
